In [1]:
import sys
import numpy as np

import geopandas as gpd
import rasterio
from rasterio.mask import mask

import time
import calendar

import pystac_client
from pystac_client.stac_api_io import APIError
from rasterio.errors import RasterioIOError
import planetary_computer
from IPython.display import clear_output
from rasterio.env import Env

sys.path.append("../utils")

In [2]:
# Change these fields as needed

# Input data filepath
filepath = "/capstone/wildfire_prep/data/PUZZLE_PIECES/inspections_master_training_geometries.geojson"

# Output data filepath
output_path = "/capstone/wildfire_prep/data/PUZZLE_PIECES/"


In [3]:
inspections = gpd.read_file(
    filepath
).drop(columns = 'apn')
inspections.head()


,inspection_id,Date,year,month,status,geometry
0,1,2019-06-05,2019,6,Compliant,"POLYGON ((-2892.357 -371905.217, -2813.179 -37..."
1,2,2019-05-09,2019,5,Compliant,"POLYGON ((-8475.394 -370860.259, -8368.416 -37..."
2,3,2019-05-09,2019,5,Compliant,"POLYGON ((-8253.618 -371183.727, -8136.129 -37..."
3,4,2019-05-09,2019,5,Compliant,"POLYGON ((-8201.866 -370989.432, -8091.086 -37..."
4,5,2019-12-17,2019,12,Compliant,"POLYGON ((-8065.883 -370931.624, -7966.144 -37..."


In [7]:
# Third Iteration of Function


def add_mean_ndvi(dataframe):
    
    # This is to avoid errors from .tifs in URLs
    env = Env(CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tif,.TIF")
    
    df_out = dataframe.copy()

    df_query = df_out.to_crs(epsg=4326)

    # Open the catalog once
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=planetary_computer.sign_inplace,
    )

    mean_vals = []

    for insp_id in df_query["inspection_id"]:
        row = df_query[df_query["inspection_id"] == insp_id].iloc[0]
        geom = row.geometry

        # Build date range
        year, month = int(row["year"]), int(row["month"])
        last_day = calendar.monthrange(year, month)[1]
        start = f"{year}-{month:02d}-01"
        end = f"{year}-{month:02d}-{last_day:02d}"

        # Search but at NaN if API Error is found
        # Search built from month and year columns in dataframe
        # Searching for imagery with <25% cloud cover. <5% wasn't catching many scenes.
        try:
            search = catalog.search(
                collections=["sentinel-2-l2a"],
                bbox=geom.bounds,
                datetime=f"{start}/{end}",
                query={"eo:cloud_cover": {"lte": 25}},
            )
            item = next(search.items(), None)

        except APIError:
            print(f"inspection_id {insp_id}: STAC API Timeout Error: skipping, NaN")
            mean_vals.append(np.nan)
            time.sleep(0.5)
            continue

        if item is None:
            print(f"inspection_id {insp_id}: no scene found, NaN")
            mean_vals.append(np.nan)
            continue

        # Get CRS of raster
        with rasterio.open(item.assets["B04"].href) as src:
            cog_crs = src.crs

        # Reproject geometry
        single = df_query[df_query["inspection_id"] == insp_id].to_crs(cog_crs)
        mask_geom = [single.geometry.iloc[0]]

        # Clip and compute NDVI, avoiding errors. If error, input NaN
        try:
            with env:
                with (
                    rasterio.open(item.assets["B04"].href) as src_red,
                    rasterio.open(item.assets["B08"].href) as src_nir,
                ):
                    red_clip, _ = mask(src_red, mask_geom, crop=True)
                    nir_clip, _ = mask(src_nir, mask_geom, crop=True)
        except (ValueError, RasterioIOError) as e:
            print(f"inspection_id {insp_id}: masking error ({e}) → NaN")
            mean_vals.append(np.nan)
            continue

        # Calculate NDVI, suppressing warnings.
        red = red_clip.astype("float32")
        nir = nir_clip.astype("float32")
        with np.errstate(divide="ignore", invalid="ignore"):
            ndvi = (nir - red) / (nir + red)

        mean_val = float(np.nanmean(ndvi))
        mean_vals.append(mean_val)
        print(f"inspection_id {insp_id}: NDVI calculated = {mean_val:.4f}")

    df_out["mean_ndvi"] = mean_vals
    return df_out


In [8]:
# Dataframe split; running the whole frame takes way too long and causes an API timeout. We'll do it in tenths.

# total rows
n = len(inspections)

# compute a chunk size so that the first 9 are equal and the last picks up any remainder
chunk_size = n // 10
remainder = n % 10

splits = []
start = 0
for i in range(10):
    extra = 1 if i < remainder else 0
    stop = start + chunk_size + extra
    splits.append(inspections.iloc[start:stop])
    start = stop

# now print out the sizes
for i, split_df in enumerate(splits, start=1):
    print(f"Split {i}: {len(split_df)} rows, {len(split_df.columns)} columns")

Split 1: 6758 rows, 6 columns
Split 2: 6758 rows, 6 columns
Split 3: 6758 rows, 6 columns
Split 4: 6758 rows, 6 columns
Split 5: 6758 rows, 6 columns
Split 6: 6758 rows, 6 columns
Split 7: 6758 rows, 6 columns
Split 8: 6758 rows, 6 columns
Split 9: 6758 rows, 6 columns
Split 10: 6758 rows, 6 columns


In [9]:
# Maybe you already processed some splits! Select the split to start from here.
start_split = 8

for i, split_df in enumerate(splits[start_split - 1 :], start=start_split):
    clear_output(wait=True)
    print(f"--- Processing split {i} of {len(splits)} ---")

    # Compute NDVI for this chunk
    df_chunk = add_mean_ndvi(split_df)

    # Report how many NaNs were produced
    n_missing = df_chunk["mean_ndvi"].isna().sum()

    # Count how many mean_ndvi values are below -0.1 (likely clouds)
    n_clouds = (df_chunk["mean_ndvi"] < -0.1).sum()
    print(
        f"Split {i}: {n_missing} missing NDVI values; "
        f"{n_clouds} values below -0.1 (likely clouds)"
    )

    # Save only the needed columns
    df_chunk = df_chunk[["inspection_id", "mean_ndvi"]]

    # Save to CSV, embedding the split number in the filename
    out_path = output_path
    df_chunk.to_csv(out_path, index=False)


--- Processing split 10 of 10 ---
inspection_id 60823: NDVI calculated = 0.2131
inspection_id 60824: NDVI calculated = 0.3109
inspection_id 60825: NDVI calculated = 0.4421
inspection_id 60826: NDVI calculated = 0.4198
inspection_id 60827: NDVI calculated = 0.4833
inspection_id 60828: NDVI calculated = 0.4280
inspection_id 60829: NDVI calculated = 0.4223
inspection_id 60830: NDVI calculated = 0.3932
inspection_id 60831: NDVI calculated = 0.4180
inspection_id 60832: NDVI calculated = 0.4686
inspection_id 60833: NDVI calculated = 0.4643
inspection_id 60834: NDVI calculated = 0.4236
inspection_id 60835: NDVI calculated = 0.4333
inspection_id 60836: NDVI calculated = 0.4663
inspection_id 60837: NDVI calculated = 0.4679
inspection_id 60838: NDVI calculated = 0.4504
inspection_id 60839: NDVI calculated = 0.4540
inspection_id 60840: NDVI calculated = 0.4228
inspection_id 60841: NDVI calculated = 0.4598
inspection_id 60842: NDVI calculated = 0.5084
inspection_id 60843: NDVI calculated = 0.4674
